In [ ]:
# --- imports ---
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from ib_insync import Stock

sys.path.append(os.path.abspath(".."))
from ibkr.Class_IBKR_IB import IBKR_IB


# ============================================================
# SETTINGS - edit these values as needed
# Keep the anchor symbol last in the list.
# ============================================================
sorted_symbols_list = ["SCHH", "USRT"]
ibkr_port = 7496
output_directory = Path("with_enhanced_prices")

lookback_period = "5 Y"
length_of_each_period = "1 day"
use_regular_trading_hours = True
prices_to_use = "TRADES"

ibkr = IBKR_IB(port=ibkr_port)


async def get_historical_closes_df(contract_list):
    close_series = []

    for contract in contract_list:
        bars = await ibkr.ib.reqHistoricalDataAsync(
            contract=contract,
            endDateTime="",  # Empty means now.
            durationStr=lookback_period,
            barSizeSetting=length_of_each_period,
            whatToShow=prices_to_use,
            useRTH=use_regular_trading_hours,
            formatDate=1,
        )

        symbol_df = pd.DataFrame(
            [(bar.date, bar.close) for bar in bars],
            columns=["date", contract.symbol],
        )
        symbol_df["date"] = pd.to_datetime(symbol_df["date"]).dt.date
        close_series.append(symbol_df.set_index("date")[contract.symbol])

    # Exclude the latest bar because it may still be incomplete.
    return pd.concat(close_series, axis=1).iloc[:-1]


async def main():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

    contract_list = []
    for symbol in sorted_symbols_list:
        contract = Stock(symbol, "SMART", "USD")
        await ibkr.ib.qualifyContractsAsync(contract)
        contract_list.append(contract)

    df = await get_historical_closes_df(contract_list)
    df.reset_index(names="to date", inplace=True)
    df.insert(0, "from date", df["to date"].shift(1))

    for symbol in sorted_symbols_list:
        from_price = f"from {symbol} price"
        to_price = f"to {symbol} price"

        df[from_price] = df[symbol].shift(1)
        df[to_price] = df[symbol]
        df[f"{symbol} cod"] = df[to_price] - df[from_price]
        df[f"{symbol} pct cod"] = np.log(df[to_price] / df[from_price])
        df.drop(columns=symbol, inplace=True)

    anchor = sorted_symbols_list[-1]
    for symbol in sorted_symbols_list:
        df[f"to {anchor} / to {symbol}"] = (
            df[f"to {anchor} price"] / df[f"to {symbol} price"]
        )

    output_directory.mkdir(parents=True, exist_ok=True)
    output_path = output_directory / f"{'_'.join(sorted_symbols_list)}.csv"
    df.to_csv(output_path)
    print(f"Saved {output_path}")
    print("finished")


await main()
